# Import

In [ ]:
import parcels
from parcels import FieldSet, JITParticle, ParticleSet, NestedField

from datetime import datetime, timedelta

import numpy as np

from pathlib import Path
import geopandas as gpd

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cmocean

import xarray as xr
import cartopy.crs as ccrs
from cartopy.crs import Geodetic, PlateCarree
from time import time
import warnings
from shapely import Polygon

warnings.simplefilter("ignore")

# Set Parameters

In [ ]:
# Parameters
RNG_seed = 123

# Base path for project data (varies per user)
base_path = "/gxfs_work/geomar/smomw122/2025_fucus-dispersal/"

# Release date (ISO format)
start_date = "2019-01-01"

# Experiment type: "surface", "bottom", or "surface_stokes"
experiment_type = "surface"

# Timesteps in minutes
calc_dt_mins = 5
output_dt_mins = 60

# What is the maximum age of the particles
max_age_days = 10

# Relative speed of the particles
relative_particle_speed = 1
# How many particles should be released each time per release cell?
particles_per_cell = 100

# Time Parameters

In [ ]:
# Get time variables from parameters
release_date = datetime.fromisoformat(start_date)
release_year = release_date.year

calc_dt = timedelta(minutes=calc_dt_mins)
output_dt = timedelta(minutes=output_dt_mins)

last_modeling_date = release_date + timedelta(days=max_age_days)
last_loaded_date = last_modeling_date + timedelta(days=1)
print(
        "release date:", release_date.date(),
        "\nlast modeling date:", last_modeling_date.date(),
        "\nlast loaded date:", last_loaded_date.date(),
)

In [ ]:
# Set random seed
np.random.seed(RNG_seed)
years = range(release_date.year, last_modeling_date.year + 1)
# Get the important dates as strings for file name and filter
release_date_str = release_date.strftime("%Y%m%d")
last_loaded_date_str = last_loaded_date.strftime("%Y%m%d")

## Set Path Variables

In [ ]:
# establish folder paths
base_path = Path(base_path)
repo_path = base_path
save_path = Path(base_path, "output/Trajectories/", str(release_year))
path_2d_fields = Path(base_path, "output/2d_fields")
path_static_files = Path("/gxfs_work/geomar/smomw122/bsh_operationalmodel_data")
path_static_fine = Path(path_static_files, "static_file_fine")
path_static_coarse = Path(path_static_files, "static_file_coarse")

In [ ]:
# 2D file suffix for the selected experiment type
file_suffix = f"_{experiment_type}.nc"

In [ ]:
# Get 2D filenames filtered for the timeframe
def get_file_list(resolution: str, suffix: str, path: Path):
    files_list = sorted(path.glob(f"c_file_{resolution}_*{suffix}"))
    # Get the position of the date in the file name
    a = files_list[0].name.find(str(release_year))
    b = a + 8
    # filter list for dates, between the start and end of the timeframe
    return list(filter(
        lambda x: release_date_str <= x.name[a:b] <= last_loaded_date_str,
        files_list
    ))

In [ ]:
def get_timestamp_from_file(fname):
    YYYYMMDDHH = fname.name.split("_")[3]
    y = YYYYMMDDHH[:4]
    m = YYYYMMDDHH[4:6]
    d = YYYYMMDDHH[6:8]
    h = YYYYMMDDHH[-2:]
    t0 = np.datetime64(f"{y}-{m}-{d}T{h}:15:00")
    return list(t0 + np.arange(4 * 6) * np.timedelta64(15 * 60, "s"))

In [ ]:
# get file list paths
current_files_fine = get_file_list("fine", file_suffix, path_2d_fields)
current_files_coarse = get_file_list("coarse", file_suffix, path_2d_fields)
print(
    "Number of files per list:", len(current_files_coarse),
    "\nlast loaded file:", current_files_coarse[-1].name,
)

In [ ]:
timestamps = [get_timestamp_from_file(cf) for cf in current_files_fine]

# Construct release locations

In [ ]:
# Read the release area file
release_area_path = Path(repo_path, "data/Fucus_location_shp")
gdf_release_area = gpd.read_file(Path(release_area_path, "REDLIST_SIS_Macrophytes.geojson"))
# Filter release area file for locations where Fucus was found and remove irrelevant columns
gdf_release_area = gdf_release_area.loc[
    (gdf_release_area.F_vesiculo != 0).values, 
    ["F_vesiculo", "geometry", "CELLID"],
].to_crs(crs=Geodetic()) # Make sure, it is in the right projection

n_release_cells = len(gdf_release_area)
n_total_particles = particles_per_cell * n_release_cells
release_time = release_date + timedelta(hours=6)

print(
        "total number of particles:", n_total_particles, 
        "\nnumber of release cells:", n_release_cells,
        "\nrelease time:", release_time,
)

In [ ]:
def relative_position_in_cell(x_rel: float, y_rel: float, cell: Polygon):
    (x0, y0), (x1, y1), (x2, y2), (x3, y3), (x4, y4) = cell.exterior.coords
    ex = (x3-x0, y3-y0)
    ey = (x1-x0, y1-y0)
    return x0 + x_rel * ex[0] + y_rel * ey[0], y0 + x_rel * ex[1] + y_rel * ey[1]

In [ ]:
# pick a random cell and a random position within cell
release_lons, release_lats, cell_IDs = list(zip(*[
    relative_position_in_cell(rand_x, rand_y, gdf_release_area.iloc[rand_cell].geometry)
    + (gdf_release_area.iloc[rand_cell].CELLID,)
    for rand_x, rand_y, rand_cell in zip(
        np.random.uniform(size=n_total_particles),
        np.random.uniform(size=n_total_particles),
        np.random.randint(0, n_release_cells, size=n_total_particles),
    )
]))

print("length of array:", len(release_lons))

# Parcels

## Custom Kernel

In [ ]:
def delete_error_particle(particle, fieldset, time):
    if particle.state >= 50:  # This captures all Errors
        particle.delete()

In [ ]:
# Establish aging for particles
def aging_kernel(particle, fieldset, time):
    particle.age_sec += particle.dt
    if particle.age_sec > fieldset.max_age_sec:
        particle.delete()

In [ ]:
def AdvectionRK4_2D_BSH(particle, fieldset, time):
    dt = particle.dt
    lat0 = particle.lat
    lon0 = particle.lon
    time0 = time

    (u1, v1) = fieldset.UV[time0, 0, lat0, lon0]
    lat1 = lat0 + v1 * 0.5 * dt
    lon1 = lon0 + u1 * 0.5 * dt
    time1 = time0 + 0.5 * dt

    (u2, v2) = fieldset.UV[time1, 0, lat1, lon1]
    lat2 = lat0 + v2 * 0.5 * dt
    lon2 = lon0 + u2 * 0.5 * dt
    time2 = time0 + 0.5 * dt

    (u3, v3) = fieldset.UV[time2, 0, lat2, lon2]
    lat3 = lat0 + v3 * dt
    lon3 = lon0 + u3 * dt
    time3 = time0 + dt

    (u4, v4) = fieldset.UV[time3, 0, lat3, lon3]
    lon4 = lon0 + (u1 + 2 * u2 + 2 * u3 + u4) / 6 * dt
    lat4 = lat0 + (v1 + 2 * v2 + 2 * v3 + v4) / 6 * dt

    particle_dlon += (lon4 - lon0) * fieldset.relative_particle_speed
    particle_dlat += (lat4 - lat0) * fieldset.relative_particle_speed

In [ ]:
custom_kernel = [
    AdvectionRK4_2D_BSH,
    # aging_kernel,
    # delete_error_particle,
]

## Fieldset

In [ ]:
# Function for constructing fieldsets
def make_fieldset(
        data_files: list, variable_ID: list,
        variable_names: list, interp_methods: list,
        timestamps: list,
        ):
        data_filenames = dict(zip(variable_ID, [data_files] * len(variable_ID)))
        data_variables = dict(zip(variable_ID, variable_names))
        data_dimensions = dict(zip(variable_ID, field_dimensions))
        interp_method = dict(zip(variable_ID, interp_methods))

        return FieldSet.from_netcdf(
                timestamps=timestamps,
                filenames=data_filenames,
                variables=data_variables,
                dimensions=data_dimensions,
                interp_method=interp_method,
                allow_time_extrapolation=False,
                gridindexingtype="nemo",
        )

In [ ]:
# Create dictionary with dimensions (2D: no depth)
dimension_dict = dict(lon="lon", lat="lat", time="time")
field_dimensions = [dimension_dict, dimension_dict]

# Prepare reading of variables
current_variable_ID = ["U", "V"]
current_variable_names = ["uvel", "vvel"]
current_interp_methods = ["cgrid_velocity", "cgrid_velocity"]

In [ ]:
# Create the fieldsets for currents
current_fieldset_fine = make_fieldset(
        data_files=current_files_fine,
        variable_ID=current_variable_ID,
        variable_names=current_variable_names,
        interp_methods=current_interp_methods,
        timestamps=timestamps,
)

In [ ]:
current_fieldset_coarse = make_fieldset(
        data_files=current_files_coarse,
        variable_ID=current_variable_ID,
        variable_names=current_variable_names,
        interp_methods=current_interp_methods,
        timestamps=timestamps,
)

### Nested fieldset

In [ ]:
# Build nested fieldset
U_nested_field = NestedField("U", [current_fieldset_fine.U, current_fieldset_coarse.U])
V_nested_field = NestedField("V", [current_fieldset_fine.V, current_fieldset_coarse.V])
nested_fieldset = FieldSet(U_nested_field, V_nested_field)
# Add the maximum age as a constant
nested_fieldset.add_constant("max_age_sec", max_age_days * 24 * 60 * 60)
nested_fieldset.add_constant("relative_particle_speed", relative_particle_speed)

## Create Particles

In [ ]:
# Establish particles and their variables
# particle_variables = (
        # "u", "v",
        ## "S", "T",
# )
sample_particle = JITParticle#.add_variables(particle_variables)
sample_particle = sample_particle.add_variable(parcels.Variable("cell_ID", initial=cell_IDs))
sample_particle = sample_particle.add_variable("age_sec", initial=0)

In [ ]:
# Create particle set
pset = ParticleSet(
        fieldset=nested_fieldset,
        pclass=sample_particle,
        lat=release_lats,
        lon=release_lons,
        time=release_time,
)

In [ ]:
# Create output filename and path
output_filename = f"Fucus_BSH_{release_date_str}_{experiment_type}_dt{output_dt_mins}min_s{relative_particle_speed}_N{n_total_particles}_seed{RNG_seed}.zarr"

output_path = Path(save_path, output_filename)
print("Output path:", output_path)

In [ ]:
from zarr.storage import MemoryStore

In [ ]:
output_path = MemoryStore()

In [ ]:
# Define Outputparameters
output_chunks = None  # (10000, int(24 * 60 / output_dt_mins)*40)

output_particle_file = pset.ParticleFile(
        name=output_path,
        outputdt=output_dt,
        chunks=output_chunks,
)

# Execute

In [ ]:
# Execute Simulation
pset.execute(
    custom_kernel,
    dt=calc_dt,
    endtime=last_modeling_date,
    output_file=output_particle_file,
    verbose_progress=True,
)

# Analysis

## Read files

In [ ]:
# Read trajectories
ds_trajectories = xr.open_zarr(output_path).compute().sortby("trajectory")
ds_trajectories

In [ ]:
ds_trajectories.time.isel(trajectory=0).min(skipna=True)
ds_trajectories.time.max(skipna=True)

In [ ]:
# Overview plot on which are alive
ds_trajectories.lon.plot()

In [ ]:
ds_trajectories.lat.diff("obs").plot.hist(bins=101)
plt.show()

In [ ]:
print(
        "First sampled time:", ds_trajectories.time.min(skipna=True).values,
        "\nLast sampled time:", ds_trajectories.time.max(skipna=True).values,
        "\n",
        "\nFirst particle:", 
        ds_trajectories.isel(trajectory=0).time.min(skipna=True).values, "-",
        ds_trajectories.isel(trajectory=0).time.max(skipna=True).values,
        "\nLast particle:", 
        ds_trajectories.isel(trajectory=-1).time.min(skipna=True).values,"-",
        ds_trajectories.isel(trajectory=-1).time.max(skipna=True).values,
        "\n",
        "\nmaximum age in days:", max_age_days, 
        "\nfirst release date:", first_release_date, 
        "\nlast release date:", last_release_date, 
        "\nlast modeling date:", last_modeling_date,
        
)

## Maps

In [ ]:
first_lat = ds_trajectories.isel(obs=0).lat
first_lon = ds_trajectories.isel(obs=0).lon

In [ ]:
last_valid_obs = ds_trajectories.obs.where(ds_trajectories.lon.notnull()).max('obs').astype(int)
last_step = ds_trajectories.isel(obs=last_valid_obs).compute()

last_lon = last_step.lon
last_lat = last_step.lat

last_step.to_dataframe().describe()

In [ ]:
traj_dead = ds_trajectories.where(first_lon == last_lon, drop=True)
traj_moving = ds_trajectories.where(first_lon != last_lon, drop=True)

In [ ]:
# open depth files and define depth cmap
depth_coarse = xr.open_dataset(path_static_coarse / "H0_file_coarse.nc").H0
depth_fine = xr.open_dataset(path_static_fine / "H0_file_fine.nc").H0
cm_deep = cmocean.cm.deep

In [ ]:
traj_lat = np.concatenate(traj_moving.lat.values)
traj_lon = np.concatenate(traj_moving.lon.values)

traj_lat = traj_lat[np.invert(np.isnan(traj_lat,))]
traj_lon = traj_lon[np.invert(np.isnan(traj_lon,))]

In [ ]:
lon_min, lon_max = (9,30)
lat_min, lat_max = (52,66)
lon_range = lon_max-lon_min
lat_range = lat_max-lat_min

In [ ]:
hist, xedges, yedges = np.histogram2d(
        x=traj_lon, y=traj_lat, 
        bins=(lon_range*4, lat_range*4), 
        range=[[lon_min, lon_max], [lat_min, lat_max]],
)

In [ ]:
fig = plt.figure(figsize=(30,18))
ax = fig.add_subplot(111, projection=PlateCarree())
hm = ax.pcolormesh(xedges, yedges, hist.T, cmap="inferno", transform=PlateCarree(), norm=LogNorm())
fig.colorbar(hm)
ax.coastlines(color="darkgrey")
ax.gridlines(draw_labels=True)
plt.title("Trajectories")
plt.xlim(lon_min, lon_max)
plt.ylim(lat_min, lat_max)
plt.show()

In [ ]:
fig = plt.figure(figsize=(30,18))
ax = fig.add_subplot(111, projection=PlateCarree())
depth_fine.plot.imshow(cmap=cm_deep, ax=ax, vmin=0, vmax=200, alpha=.8)
depth_coarse.plot.imshow(cmap=cm_deep, ax=ax, add_colorbar=False, vmin=0, vmax=200, alpha=.8)

ax.scatter(
        x=ds_trajectories.lon, y=ds_trajectories.lat,
        c=ds_trajectories.cell_ID, cmap="rainbow",
        s=10,
)
ax.coastlines(color="darkgrey")
ax.gridlines(draw_labels=True)
plt.title("Trajectories")
plt.xlim(9,30)
plt.ylim(52.5,66)
plt.show()

In [ ]:
fig = plt.figure(figsize=(30,18))
ax = fig.add_subplot(111, projection=PlateCarree())
depth_fine.plot.imshow(cmap=cm_deep, ax=ax, vmin=0, vmax=200, alpha=.8)
depth_coarse.plot.imshow(cmap=cm_deep, ax=ax, add_colorbar=False, vmin=0, vmax=200, alpha=.8)

ax.scatter(traj_dead.isel(obs=0).lon, traj_dead.isel(obs=0).lat, c="r", s=10)
ax.scatter(traj_moving.isel(obs=0).lon, traj_moving.isel(obs=0).lat, c="k", s=10)
ax.coastlines(color="darkgrey")
ax.gridlines(draw_labels=True)
plt.title("First observation")
plt.xlim(9,30)
plt.ylim(52.5,66)
plt.show()

In [ ]:
fig = plt.figure(figsize=(30,18))
ax = fig.add_subplot(111, projection=PlateCarree())
depth_fine.plot.imshow(cmap=cm_deep, ax=ax, vmin=0, vmax=200, alpha=.8)
depth_coarse.plot.imshow(cmap=cm_deep, ax=ax, add_colorbar=False, vmin=0, vmax=200, alpha=.8)

ax.scatter(traj_dead.isel(obs=0).lon, traj_dead.isel(obs=0).lat, c="r", s=10)
ax.scatter(traj_moving.isel(obs=0).lon, traj_moving.isel(obs=0).lat, c="k", s=10)
ax.coastlines(color="darkgrey")
ax.gridlines(draw_labels=True)
plt.title("First observation")
plt.xlim(16.5,22)
plt.ylim(59.5,64)
plt.show()

In [ ]:
fig = plt.figure(figsize=(30,18))
ax = fig.add_subplot(111, projection=PlateCarree())
depth_fine.plot.imshow(cmap=cm_deep, ax=ax, vmin=0, vmax=200, alpha=.8)
depth_coarse.plot.imshow(cmap=cm_deep, ax=ax, add_colorbar=False, vmin=0, vmax=200, alpha=.8)

ax.scatter(first_lon, first_lat, c="darkred", s=10)
ax.scatter(last_lon, last_lat, c="orange", s=10)
ax.coastlines(color="darkgrey")
ax.gridlines(draw_labels=True)
plt.title("First observation")
plt.xlim(16.5,22)
plt.ylim(59.5,64)
plt.show()

In [ ]:
ganz_tot = np.unique(traj_dead.isel(obs=0).cell_ID.values)[
        np.invert(
                np.in1d(
                        np.unique(traj_dead.isel(obs=0).cell_ID.values),
                        np.unique(traj_moving.isel(obs=0).cell_ID.values)
                )
        )
]
len(ganz_tot)